**Implementing a Simple GCN**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    def __init__(self, input_size: int, output_size: int, A: torch.Tensor):
        super(GCNLayer, self).__init__()
        self.input_size = input_size
        self.output_size = output_size
        self.A = A
        self.A_hat = self.A + torch.eye(self.A.size(0))
        self.ones = torch.ones(input_size, input_size)
        self.D = torch.matmul(self.A.float(), self.ones.float())
        self.D = torch.diag(self.D)
        self.D = torch.diag_embed(self.D)
        self.D_neg_sqrt = torch.diag_embed(torch.diag(torch.pow(self.D, -0.5)))
        self.W = nn.Parameter(torch.rand(input_size, output_size))

    def forward(self, X: torch.Tensor):
        support_1 = torch.matmul(self.D_neg_sqrt, torch.matmul(self.A_hat, self.D_neg_sqrt))
        support_2 = torch.matmul(support_1, torch.matmul(X, self.W))
        H = F.relu(support_2)

        return H

In [2]:
input_size = 3
output_size = 2
A = torch.tensor([[1., 0., 0.],
                  [0., 1., 1.],
                  [0., 1., 1.]])

gcn_layer = GCNLayer(input_size, output_size, A)

X = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.],
                  [7., 8., 9.]])

output = gcn_layer(X)
print(output)

tensor([[ 5.0149,  6.4111],
        [11.9042, 14.0186],
        [13.9399, 16.3212]], grad_fn=<ReluBackward0>)
